# CatBoost Training (5-Fold Cross Validation)

Train the metadata-based CatBoost model used in the ISIC 2024 Skin Cancer Detection ensemble. The notebook performs five-fold Stratified Group Cross Validation, computes ROC-AUC and pAUC, and saves the trained models and Out-of-Fold predictions.

## Configuration

Define dataset paths, cross-validation settings, and CatBoost hyperparameters used throughout the notebook.

In [ ]:
# ==========================================================
# Paths
# ==========================================================

DATA_PATH = "../data/train-metadata.csv"

MODEL_DIR = "../5fold-cv-models"

FEATURES_PATH = f"{MODEL_DIR}/features.json"
OOF_PATH = f"{MODEL_DIR}/oof_predictions.csv"
FOLD_RESULTS_PATH = f"{MODEL_DIR}/fold_results.csv"

# ==========================================================
# Cross Validation
# ==========================================================

N_FOLDS = 5
SEED = 42

# ==========================================================
# CatBoost Hyperparameters
# ==========================================================

ITERATIONS = 1000
LEARNING_RATE = 0.05
DEPTH = 7
EARLY_STOPPING = 100
AUTO_CLASS_WEIGHTS = "Balanced"
EVAL_METRIC = "AUC"
VERBOSE = 100

## Imports

Import the libraries and project modules required for preprocessing, training, evaluation, and saving outputs.

In [ ]:
import numpy as np
import pandas as pd
import sys

from sklearn.model_selection import StratifiedGroupKFold
from pathlib import Path

sys.path.append(
    "../src/cat/"
)

from cat_dataset import load_data, prepare_data
from cat_train import train_fold
from cat_metrics import evaluate
from cat_utils import (
    save_features,
    save_model,
    save_oof,
    save_fold_results,
)

## Dataset Preparation

Load the ISIC 2024 metadata, perform preprocessing and feature engineering, and identify categorical features for CatBoost.

In [3]:
print("Loading dataset...")

df = load_data(DATA_PATH)

X, y, groups, cat_features = prepare_data(df)

print(X.shape)
print(y.value_counts())

Loading dataset...
Loading dataset...
(401059, 39)
target
0    400666
1       393
Name: count, dtype: int64


**Save Feature Information**

Store the feature names and categorical feature list so that the inference pipeline can reproduce the same preprocessing later.

In [4]:
save_features(
    features=X.columns.tolist(),
    categorical_features=cat_features,
    path=FEATURES_PATH,
)

## Cross Validation Setup

Use Stratified Group K-Fold to preserve both the class distribution and the patient grouping during validation.

In [5]:
cv = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED,
)

oof_predictions = np.zeros(len(df))

fold_results = []

## Training Loop

Train one CatBoost model per fold and evaluate it on the corresponding validation split.

In [6]:
for fold, (train_idx, valid_idx) in enumerate(
    cv.split(X, y, groups)
):

    print("=" * 60)
    print(f"Fold {fold}")
    print("=" * 60)

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    train_results = train_fold(
        X_train=X_train,
        y_train=y_train,
        X_valid=X_valid,
        y_valid=y_valid,
        cat_features=cat_features,
        iterations=ITERATIONS,
        learning_rate=LEARNING_RATE,
        depth=DEPTH,
        early_stopping=EARLY_STOPPING,
        auto_class_weights=AUTO_CLASS_WEIGHTS,
        eval_metric=EVAL_METRIC,
        verbose=VERBOSE,
        seed=SEED,
    )

    metric_results = evaluate(
        y_true=y_valid,
        y_pred=train_results["predictions"],
    )

    save_model(
        model=train_results["model"],
        path=f"{MODEL_DIR}/fold_{fold}.cbm",
    )

    oof_predictions[valid_idx] = train_results["predictions"]

    fold_results.append({
        "fold": fold,
        "best_iteration": train_results["best_iteration"],
        "roc_auc": metric_results["roc_auc"],
        "pauc": metric_results["pauc"],
    })

    print(
        f"Fold {fold}: "
        f"pAUC={metric_results['pauc']:.5f}"
    )

Fold 0
0:	test: 0.8765929	best: 0.8765929 (0)	total: 122ms	remaining: 2m 1s
100:	test: 0.9388411	best: 0.9490546 (48)	total: 6.08s	remaining: 54.1s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9490545686
bestIteration = 48

Shrink model to first 49 iterations.
Fold 0: pAUC=0.87658
Fold 1
0:	test: 0.8896344	best: 0.8896344 (0)	total: 65.6ms	remaining: 1m 5s
100:	test: 0.9189244	best: 0.9226867 (75)	total: 5.95s	remaining: 53s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9226867064
bestIteration = 75

Shrink model to first 76 iterations.
Fold 1: pAUC=0.85401
Fold 2
0:	test: 0.8889961	best: 0.8889961 (0)	total: 69.3ms	remaining: 1m 9s
100:	test: 0.9469802	best: 0.9471919 (95)	total: 6.05s	remaining: 53.8s
200:	test: 0.9452558	best: 0.9477225 (117)	total: 11.5s	remaining: 45.5s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9477224536
bestIteration = 117

Shrink model to first 118 iterations.
Fold 2: pAUC=0.89843
Fold 3


## Final Evaluation

Compute the overall Out-of-Fold ROC-AUC and pAUC using the predictions collected from all validation folds.

In [7]:
print("\nOverall OOF evaluation")

overall_metrics = evaluate(
    y_true=y,
    y_pred=oof_predictions,
)

print(
    f"Mean fold pAUC: "
    f"{np.mean([r['pauc'] for r in fold_results]):.5f}"
)

print(
    f"Std fold pAUC: "
    f"{np.std([r['pauc'] for r in fold_results]):.5f}"
)

print(
    f"Overall ROC-AUC: "
    f"{overall_metrics['roc_auc']:.5f}"
)

print(
    f"Overall pAUC: "
    f"{overall_metrics['pauc']:.5f}"
)


Overall OOF evaluation
Mean fold pAUC: 0.86233
Std fold pAUC: 0.02229
Overall ROC-AUC: 0.92600
Overall pAUC: 0.86093


## Save Outputs

Save the Out-of-Fold predictions and the fold-wise metrics for later use in stacking and analysis.

In [8]:
save_oof(
    dataframe=df,
    targets=y,
    predictions=oof_predictions,
    path=OOF_PATH,
)

save_fold_results(
    fold_results,
    FOLD_RESULTS_PATH,
)

## Results

Display the fold-wise metrics as a table for quick inspection.

In [9]:
pd.DataFrame(fold_results)

,fold,best_iteration,roc_auc,pauc
0,0,48,0.949055,0.876583
1,1,75,0.922687,0.854006
2,2,117,0.947722,0.898432
3,3,74,0.920343,0.837722
4,4,126,0.913094,0.844903
